In [0]:
from pyspark.sql import functions as F

orig = spark.table("workspace.mortgage_risk.bronze_origination")
perf = spark.table("workspace.mortgage_risk.bronze_performance")

# Find missing loan IDs and reporting months.
missing_orig = orig.filter(
    F.col("loan_id").isNull()
    | (F.trim(F.col("loan_id")) == "")
)

missing_perf = perf.filter(
    F.col("loan_id").isNull()
    | (F.trim(F.col("loan_id")) == "")
    | F.col("period").isNull()
    | (F.trim(F.col("period")) == "")
)

# Find duplicate keys.
duplicate_orig = (
    orig.groupBy("loan_id")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_perf = (
    perf.groupBy("loan_id", "period")
    .count()
    .filter(F.col("count") > 1)
)

# Find performance records without matching origination.
perf_without_orig = perf.join(
    orig.select("loan_id"),
    on="loan_id",
    how="left_anti"
)

# Stop the job if any check fails.
assert missing_orig.limit(1).count() == 0, "Missing origination loan IDs."
assert missing_perf.limit(1).count() == 0, "Missing performance keys."
assert duplicate_orig.limit(1).count() == 0, "Duplicate origination IDs."
assert duplicate_perf.limit(1).count() == 0, "Duplicate loan-month records."
assert perf_without_orig.limit(1).count() == 0, "Missing origination matches."

print("Silver input checks passed.")

Silver input checks passed.


In [0]:
#Load the Bronze origination table
from pyspark.sql import functions as F

orig = spark.table("workspace.mortgage_risk.bronze_origination")

display(
    orig.select(
        "loan_id", "fico", "ltv", "dti",
        "original_upb", "first_payment_date"
    ).limit(5)
)

loan_id,fico,ltv,dti,original_upb,first_payment_date
F16Q10000006,686,65,41,125000,201604
F16Q10000027,774,95,30,44000,201603
F16Q10000048,808,56,15,212000,201604
F16Q10000054,784,66,42,417000,201603
F16Q10000076,798,28,22,40000,201603


In [0]:
#Remove surrounding spaces and turn empty fields into NULL
metadata_columns = {
    "raw_line", "source_file", "source_path",
    "source_vintage", "ingested_at"
}

normalized = orig.select(
    *[
        (
            F.when(F.trim(F.col(c)) == "", F.lit(None))
             .otherwise(F.trim(F.col(c)))
             .alias(c)
        )
        if c not in metadata_columns
        else F.col(c)
        for c in orig.columns
    ]
)

display(normalized.select("loan_id", "fico", "msa").limit(5))

loan_id,fico,msa
F16Q10000006,686,39820
F16Q10000027,774,29620
F16Q10000048,808,30700
F16Q10000054,784,null
F16Q10000076,798,47664


In [0]:
#Replace unavailable-value codes
unavailable_codes = {
    "fico": "9999",
    "vantage_score": "9999",
    "mi_pct": "999",
    "ltv": "999",
    "cltv": "999",
    "dti": "999",
    "units": "99",
    "borrowers": "99",
    "first_time_homebuyer": "9",
    "occupancy": "9",
    "channel": "9",
    "property_type": "99",
    "purpose": "9",
    "postal_code": "000",
    "valuation_method": "7"
}

for column, code in unavailable_codes.items():
    normalized = normalized.withColumn(
        column,
        F.when(F.col(column) == code, F.lit(None))
         .otherwise(F.col(column))
    )

In [0]:
#Define the numeric conversions
numeric_types = {
    "fico": "INT",
    "vantage_score": "INT",
    "mi_pct": "DOUBLE",
    "units": "INT",
    "cltv": "DOUBLE",
    "dti": "DOUBLE",
    "original_upb": "DECIMAL(18,2)",
    "ltv": "DOUBLE",
    "original_rate": "DOUBLE",
    "original_term": "INT",
    "borrowers": "INT"
}

conversions = {
    column: F.expr(f"try_cast(`{column}` AS {dtype})")
    for column, dtype in numeric_types.items()
}

In [0]:
#Define the date conversions
for column in ["first_payment_date", "maturity_date"]:
    conversions[column] = F.when(
        F.col(column).rlike(r"^\d{6}$"),
        F.expr(
            f"""try_cast(
                concat(
                    substring(`{column}`, 1, 4), '-',
                    substring(`{column}`, 5, 2), '-01'
                ) AS DATE
            )"""
        )
    )

In [0]:
#Check that the conversions work (Did a value exist before conversion but become NULL because conversion failed)
conversion_check = normalized.agg(
    *[
        F.sum(
            F.when(
                F.col(column).isNotNull() & expression.isNull(),
                1
            ).otherwise(0)
        ).alias(column)
        for column, expression in conversions.items()
    ]
)

display(conversion_check)

failures = conversion_check.first().asDict()

assert all(value == 0 for value in failures.values()), \
    f"Unexpected conversion failures: {failures}"

fico,vantage_score,mi_pct,units,cltv,dti,original_upb,ltv,original_rate,original_term,borrowers,first_payment_date,maturity_date
0,0,0,0,0,0,0,0,0,0,0,0,0


In [0]:
#Apply the conversions
silver_orig = normalized.select(*[conversions[c].alias(c)
        if c in conversions
        else F.col(c)
        for c in normalized.columns
    ]
)

silver_orig.printSchema()

display(
    silver_orig.select(
        "loan_id", "fico", "ltv", "dti",
        "original_upb", "first_payment_date"
    ).limit(5)
)

root
 |-- fico: integer (nullable = true)
 |-- first_payment_date: date (nullable = true)
 |-- first_time_homebuyer: string (nullable = true)
 |-- maturity_date: date (nullable = true)
 |-- msa: string (nullable = true)
 |-- mi_pct: double (nullable = true)
 |-- units: integer (nullable = true)
 |-- occupancy: string (nullable = true)
 |-- cltv: double (nullable = true)
 |-- dti: double (nullable = true)
 |-- original_upb: decimal(18,2) (nullable = true)
 |-- ltv: double (nullable = true)
 |-- original_rate: double (nullable = true)
 |-- channel: string (nullable = true)
 |-- prepayment_penalty: string (nullable = true)
 |-- amortization_type: string (nullable = true)
 |-- state: string (nullable = true)
 |-- property_type: string (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- loan_id: string (nullable = true)
 |-- purpose: string (nullable = true)
 |-- original_term: integer (nullable = true)
 |-- borrowers: integer (nullable = true)
 |-- seller: string (nullable = 

loan_id,fico,ltv,dti,original_upb,first_payment_date
F16Q10000006,686,65.0,41.0,125000.00,2016-04-01
F16Q10000027,774,95.0,30.0,44000.00,2016-03-01
F16Q10000048,808,56.0,15.0,212000.00,2016-04-01
F16Q10000054,784,66.0,42.0,417000.00,2016-03-01
F16Q10000076,798,28.0,22.0,40000.00,2016-03-01


In [0]:
(
    silver_orig.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.mortgage_risk.silver_origination")
)

print("Silver origination table refreshed.")

Silver origination table refreshed.


In [0]:
#### Now clean the monthly performance ###
# Raw status	Meaning	Numeric delinquency
#"00"	Current or less than 30 days overdue	0
#"01"	30–59 days overdue	                    1
#"02"	60–89 days overdue	                    2
#"03"	90–119 days overdue	                    3
#"RA"	REO acquisition	                       NULL    retain separate flag
#"XX"	Status unavailable	                   NULL retain unknown flag

In [0]:
#Load the data and handle missing values
from pyspark.sql import functions as F

perf_raw = spark.table("workspace.mortgage_risk.bronze_performance")

metadata_columns = [
    "raw_line",
    "source_file",
    "source_path",
    "source_vintage",
    "ingested_at"
]

perf_clean = perf_raw

# Clean the data fields, but leave the source information unchanged.
for column in perf_raw.columns:
    if column in metadata_columns:
        continue

    value = F.trim(F.col(column))

    perf_clean = perf_clean.withColumn(
        column,
        F.when(value == "", None).otherwise(value)
    )

# Freddie Mac uses 999 when estimated LTV is unavailable.
perf_clean = perf_clean.withColumn(
    "estimated_ltv",
    F.when(F.col("estimated_ltv") == "999", None)
     .otherwise(F.col("estimated_ltv"))
)

display(
    perf_clean.select(
        "loan_id",
        "period",
        "current_upb",
        "delinquency",
        "estimated_ltv"
    ).limit(5)
)

loan_id,period,current_upb,delinquency,estimated_ltv
F20Q10000154,202002,399000.00,00,58
F20Q10000154,202003,399000.00,00,57
F20Q10000154,202004,398000.00,00,56
F20Q10000154,202005,398000.00,00,55
F20Q10000154,202006,397000.00,00,55


In [0]:
# Prepare the conversions
numeric_types = {
    "current_upb": "DECIMAL(18,2)",
    "deferred_upb": "DECIMAL(18,2)",
    "interest_bearing_upb": "DECIMAL(18,2)",
    "current_rate": "DOUBLE",
    "estimated_ltv": "DOUBLE",
    "loan_age": "INT",
    "remaining_term": "INT"
}

conversions = {}

for column, data_type in numeric_types.items():
    conversions[column] = F.expr(
        f"try_cast(`{column}` AS {data_type})"
    )


# Turn a YYYYMM value into a date, using the first day of the month.
def parse_month(column):
    date_text = F.concat(
        F.substring(F.col(column), 1, 4),
        F.lit("-"),
        F.substring(F.col(column), 5, 2),
        F.lit("-01")
    )

    # Build a temporary date expression without relying on strict parsing.
    date_value = F.expr(
        f"""try_cast(
            concat(
                substring(`{column}`, 1, 4), '-',
                substring(`{column}`, 5, 2), '-01'
            ) AS DATE
        )"""
    )

    return F.when(
        F.col(column).rlike(r"^\d{6}$"),
        date_value
    )


date_columns = [
    "period",
    "zero_balance_date",
    "defect_date",
    "ddlpi"
]

for column in date_columns:
    conversions[column] = parse_month(column)

In [0]:
# Check for unexpected values before applying the conversions
checks = []

for column, converted_value in conversions.items():
    conversion_failed = (
        F.col(column).isNotNull()
        & converted_value.isNull()
    )

    failure_count = F.sum(
        F.when(conversion_failed, 1).otherwise(0)
    ).alias(column)

    checks.append(failure_count)

conversion_results = perf_clean.agg(*checks)
display(conversion_results)

failure_counts = conversion_results.first().asDict()

for column, count in failure_counts.items():
    assert count == 0, f"{column} has {count} conversion failures."


# Delinquency should be two digits, RA, XX, or missing.
status = F.col("delinquency")

valid_status = (
    status.rlike(r"^\d{2}$")
    | status.isin("RA", "XX")
)

unexpected_statuses = perf_clean.filter(
    status.isNotNull() & ~valid_status
)

assert unexpected_statuses.limit(1).count() == 0, \
    "Found an unexpected delinquency code."

print("Conversion and delinquency checks passed.")

current_upb,deferred_upb,interest_bearing_upb,current_rate,estimated_ltv,loan_age,remaining_term,period,zero_balance_date,defect_date,ddlpi
0,0,0,0,0,0,0,0,0,0,0


Conversion and delinquency checks passed.


In [0]:
# Apply the conversions and create the status columns
silver_perf = perf_clean

for column, converted_value in conversions.items():
    silver_perf = silver_perf.withColumn(column, converted_value)


# Keep the original delinquency code and add its numeric version.
silver_perf = silver_perf.withColumn(
    "delinquency_months",
    F.when(
        F.col("delinquency").rlike(r"^\d{2}$"),
        F.expr("try_cast(delinquency AS INT)")
    )
)

silver_perf = silver_perf.withColumn(
    "is_reo",
    F.when(F.col("delinquency") == "RA", 1).otherwise(0)
)

unknown_status = (
    F.col("delinquency").isNull()
    | (F.col("delinquency") == "XX")
)

silver_perf = silver_perf.withColumn(
    "delinquency_unknown",
    F.when(unknown_status, 1).otherwise(0)
)

serious_event = (
    (F.col("delinquency_months") >= 3)
    | (F.col("is_reo") == 1)
)

# This describes the current reporting month, not the future 12-month label.
silver_perf = silver_perf.withColumn(
    "serious_delinquency_observed",
    F.when(F.col("delinquency_unknown") == 1, F.lit(None).cast("int"))
     .when(serious_event, 1)
     .otherwise(0)
)

display(
    silver_perf.select(
        "loan_id",
        "period",
        "current_upb",
        "delinquency",
        "delinquency_months",
        "is_reo",
        "delinquency_unknown",
        "serious_delinquency_observed"
    ).limit(10)
)

loan_id,period,current_upb,delinquency,delinquency_months,is_reo,delinquency_unknown,serious_delinquency_observed
F20Q10000154,2020-02-01,399000.00,00,0,0,0,0
F20Q10000154,2020-03-01,399000.00,00,0,0,0,0
F20Q10000154,2020-04-01,398000.00,00,0,0,0,0
F20Q10000154,2020-05-01,398000.00,00,0,0,0,0
F20Q10000154,2020-06-01,397000.00,00,0,0,0,0
F20Q10000154,2020-07-01,396000.00,00,0,0,0,0
F20Q10000154,2020-08-01,396000.00,00,0,0,0,0
F20Q10000154,2020-09-01,0.00,00,0,0,0,0
F20Q10000157,2020-02-01,78000.00,00,0,0,0,0
F20Q10000157,2020-03-01,78000.00,00,0,0,0,0


In [0]:
(
    silver_perf.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.mortgage_risk.silver_performance")
)

print("Silver performance table refreshed.")

Silver performance table refreshed.


In [0]:
# check for missing months in each loan’s payment history
from pyspark.sql import functions as F
from pyspark.sql.window import Window

perf = spark.table("workspace.mortgage_risk.silver_performance")

# Check each loan's months in chronological order
loan_window = Window.partitionBy("loan_id").orderBy("period")

history = perf.select("loan_id", "period")

history = history.withColumn(
    "previous_month",
    F.lag("period").over(loan_window)
)

history = history.withColumn(
    "months_apart",
    F.months_between("period", "previous_month").cast("int")
)

# A difference greater than one means some months are missing
gaps = history.filter(F.col("months_apart") > 1)

gaps = gaps.withColumn(
    "missing_months",
    F.col("months_apart") - 1
)

display(
    gaps.select(
        "loan_id",
        "previous_month",
        F.col("period").alias("next_observed_month"),
        "missing_months"
    ).orderBy("loan_id", "previous_month")
)

loan_id,previous_month,next_observed_month,missing_months
F17Q40045268,2021-04-01,2021-07-01,2
